# 06 — Imbalance Handling (Experiments B, C, D)

Compare class weighting, random oversampling, and SMOTE — all applied **only to the training folds** via leakage-safe pipelines.

In [ ]:
import sys
from pathlib import Path

# Make the project root importable (so `import src...` works).
ROOT = Path.cwd()
if (ROOT / 'src').exists() is False and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.utils import config
config.ensure_directories()

In [ ]:
from src.data import load_data, preprocessing
from src.models import baseline, train

df = load_data.load_dataset()
X, y = preprocessing.separate_features_target(df)
X_train, X_test, y_train, y_test = preprocessing.split_data(X, y)

## Run each strategy for every model

`class_weight` uses the weighted estimators; resampling strategies use the default estimators inside an imbalanced-learn pipeline.

In [ ]:
results = []
strategy_models = {
    'class_weight': baseline.get_class_weighted_models(),
    'random_oversample': baseline.get_models(),
    'smote': baseline.get_models(),
}
for strategy, models in strategy_models.items():
    for name, estimator in models.items():
        results.append(train.run_experiment(
            model_name=name, estimator=estimator, strategy=strategy,
            X_train=X_train, X_test=X_test,
            y_train=y_train, y_test=y_test,
        ))
results_df = train.results_to_frame(results)
results_df

## Save per-strategy metrics

In [ ]:
for strategy in results_df['strategy'].unique():
    subset = results_df[results_df['strategy'] == strategy]
    out = config.METRICS_DIR / f'{strategy}_results.csv'
    subset.to_csv(out, index=False)
    print('saved ->', out)

## Before/after resampling distribution (illustrative)

In [ ]:
from imblearn.over_sampling import SMOTE
from src.analysis import imbalance
from src.visualization import plots

# Illustration on TRAINING data only. In experiments this happens
# inside the pipeline; here we show the effect explicitly.
num = preprocessing.identify_numerical_features(X_train)
cat = preprocessing.identify_categorical_features(X_train)
pre = preprocessing.build_preprocessor(num, cat).fit(X_train)
X_tr_enc = pre.transform(X_train)
_, y_res = SMOTE(random_state=config.RANDOM_STATE).fit_resample(X_tr_enc, y_train)
comp = imbalance.compare_distributions(y_train, y_res)
plots.plot_resampling_comparison(comp, save_path='smote_before_after.png')
plt.show()
comp